In [24]:
import pandas as pd
import numpy as np
from math import ceil
from collections import defaultdict
from itertools import combinations
import mlxtend

pd.set_option('display.max_colwidth', None)

In [2]:
RUTA_DATASET = "dataset/data_secretariado.csv"

In [3]:
df = pd.read_csv(RUTA_DATASET, dtype=str)

print("Dimensiones del dataset:")
print(df.shape)

Dimensiones del dataset:
(133887, 11)


# **Ejercicio 1:** Implementación de algoritmo de asociación

## Preparación de los datos para Apriori

El dataset ya viene limpio, por lo que no se realizará una limpieza profunda.
Sin embargo, sí se requieren algunas transformaciones para que los datos sean compatibles
con el algoritmo Apriori.

### Transformaciones realizadas

- **`ID_VICTIMA` se elimina** porque es un identificador único por registro; incluirlo
  generaría ítems que nunca se repiten y no aportaría ningún patrón útil.

- **`ORIGEN_REPORTE` se omite** porque está altamente correlacionado con `ENTIDAD`,
  lo que generaría reglas triviales del tipo "reporte de Puebla → entidad Puebla".

- **Las fechas se discretizan por sexenio**. Usar la fecha completa (`2025-09-28 4:40:00`)
  generaría miles de valores únicos, haciendo imposible encontrar patrones frecuentes.
  Agrupar por sexenio permite capturar tendencias políticas y administrativas relevantes
  en el contexto mexicano, donde los cambios de gobierno pueden estar asociados a
  variaciones en el registro y atención de casos de desaparición.

- **`ESTATUS_VICTIMA` se elimina** porque el 94.27% de los registros (tras filtrar
  los valores confidenciales) corresponden a `DESAPARECIDA`. Al ser un ítem casi
  universal, generaría reglas trivialmente verdaderas con confianza artificialmente
  alta, sin aportar información real sobre el fenómeno.

- **Cada fila se convierte en una transacción**, es decir, un conjunto de ítems del tipo
  `COLUMNA=valor` (por ejemplo: `SEXO=MUJER`, `MES_DESAPARICION=09`, `ENTIDAD=PUEBLA`).

In [4]:
data = df.copy()

# ─── Parámetro de control ────────────────────────────────────────────────────
# Si es False, se eliminan filas donde las columnas clave sean CONFIDENCIAL.
# Esto reduce el dataset pero mejora la calidad de las reglas obtenidas,
# ya que los registros confidenciales aportan muy poca información útil.
INCLUIR_CONFIDENCIALES = False

if not INCLUIR_CONFIDENCIALES:
    columnas_confidenciales = ["SEXO", "ESTATUS_VICTIMA", "MUNICIPIO"]
    for col in columnas_confidenciales:
        data = data[data[col] != "CONFIDENCIAL"]

# ─── Conversión de fechas ─────────────────────────────────────────────────────
# Convertimos las columnas de fecha a tipo datetime.
# errors="coerce" convierte valores inválidos o vacíos en NaT (Not a Time),
# lo que nos permite manejarlos de forma controlada más adelante.
for col_str, col_dt in [
    ("FECHA_DESAPARICION", "FECHA_DESAPARICION_DT"),
    ("FECHA_REGISTRO",     "FECHA_REGISTRO_DT"),
    ("FECHA_NACIMIENTO",   "FECHA_NACIMIENTO_DT"),
]:
    data[col_dt] = pd.to_datetime(data[col_str], errors="coerce")

# ─── Discretización por sexenio ───────────────────────────────────────────────
# Usar la fecha completa generaría demasiados valores únicos, lo que haría
# imposible encontrar patrones frecuentes. Agrupar por sexenio permite capturar
# tendencias político-administrativas relevantes en el contexto mexicano,
# donde los cambios de gobierno pueden asociarse a variaciones en el registro
# y atención de casos de desaparición.

SEXENIOS_BINS   = [1994, 2000, 2006, 2012, 2018, 2024, 2030]
SEXENIOS_LABELS = [
    "Zedillo (1994-2000)",
    "Fox (2000-2006)",
    "Calderón (2006-2012)",
    "Peña Nieto (2012-2018)",
    "AMLO (2018-2024)",
    "Sheinbaum (2024-2030)",
]

def asignar_sexenio(serie_dt):
    """Recibe una Serie datetime y devuelve el sexenio correspondiente."""
    anio = serie_dt.dt.year
    return (
        pd.cut(anio, bins=SEXENIOS_BINS, labels=SEXENIOS_LABELS, right=False)
        .astype(str)
        .replace("nan", "DESCONOCIDO")
    )

data["SEXENIO_DESAPARICION"] = asignar_sexenio(data["FECHA_DESAPARICION_DT"])
data["SEXENIO_REGISTRO"]     = asignar_sexenio(data["FECHA_REGISTRO_DT"])

# ─── Mes de desaparición ──────────────────────────────────────────────────────
# Mantenemos el mes para detectar posibles patrones de estacionalidad,
# algo que el sexenio por sí solo no captura.
data["MES_DESAPARICION"] = (
    data["FECHA_DESAPARICION_DT"]
    .dt.month
    .astype("Int64")
    .astype(str)
    .str.zfill(2)
    .replace("<NA>", "DESCONOCIDO")
)

# ─── Grupo de edad ────────────────────────────────────────────────────────────
# Calculamos la edad aproximada al momento de la desaparición.
# Dividimos entre 365.25 para considerar los años bisiestos.
edad = (data["FECHA_DESAPARICION_DT"] - data["FECHA_NACIMIENTO_DT"]).dt.days / 365.25

# Descartamos edades fuera de un rango biológicamente plausible.
edad = edad.where((edad >= 0) & (edad <= 120))

bins_edad   = [-np.inf, 11, 17, 29, 44, 59, np.inf]
labels_edad = ["0-11", "12-17", "18-29", "30-44", "45-59", "60+"]
data["GRUPO_EDAD"] = (
    pd.cut(edad, bins=bins_edad, labels=labels_edad)
    .astype(str)
    .replace("nan", "DESCONOCIDO")
)

# ─── Selección de columnas para Apriori ───────────────────────────────────────
# Solo incluimos variables categóricas o ya discretizadas.
# Se excluye ORIGEN_REPORTE porque está altamente correlacionado con ENTIDAD
# y generaría reglas triviales del tipo "reporte de Puebla → entidad Puebla".
# Se excluye ESTATUS_VICTIMA porque el 94.27% de los registros (tras filtrar
# confidenciales) corresponden a DESAPARECIDA, lo que lo convierte en un ítem
# casi universal. Su presencia generaría reglas trivialmente verdaderas con
# confianza artificialmente alta, sin aportar información real sobre el fenómeno.
# Se excluyen las fechas crudas y los IDs.
COLUMNAS_ITEMS = [
    "SEXO",
    #"ESTATUS_VICTIMA", # descomentar si INCLUIR_CONFIDENCIALES = true
    "ENTIDAD",
    "MUNICIPIO",
    "SEXENIO_DESAPARICION",
    "SEXENIO_REGISTRO",
    "MES_DESAPARICION",
    "GRUPO_EDAD",
]

data_apriori = data[COLUMNAS_ITEMS].copy()

print("Dimensiones después de preparar los datos:")
print(data_apriori.shape)
data_apriori.head()

Dimensiones después de preparar los datos:
(84738, 7)


,SEXO,ENTIDAD,MUNICIPIO,SEXENIO_DESAPARICION,SEXENIO_REGISTRO,MES_DESAPARICION,GRUPO_EDAD
1,HOMBRE,SINALOA,CONCORDIA,Sheinbaum (2024-2030),Sheinbaum (2024-2030),09,45-59
5,HOMBRE,PUEBLA,PUEBLA,Sheinbaum (2024-2030),Sheinbaum (2024-2030),09,0-11
7,MUJER,PUEBLA,PUEBLA,Sheinbaum (2024-2030),Sheinbaum (2024-2030),09,30-44
11,HOMBRE,MICHOACÁN,TINGÜINDÍN,AMLO (2018-2024),AMLO (2018-2024),03,30-44
14,HOMBRE,HIDALGO,PACHUCA DE SOTO,Sheinbaum (2024-2030),Sheinbaum (2024-2030),09,0-11


Notemos que algunas reglas son esperadas debido a relaciones jerárquicas entre variables, por ejemplo MUNICIPIO y ENTIDAD.

In [5]:
def crear_transacciones(df_items):
    """
    Convierte un DataFrame en una lista de transacciones para Apriori.

    En minería de reglas de asociación, una "transacción" es un conjunto de ítems.
    Cada fila del DataFrame se convierte en una lista de ítems con el formato
    COLUMNA=valor (por ejemplo: SEXO=MUJER, ENTIDAD=PUEBLA).

    Parámetros
    ----------
    df_items : pd.DataFrame
        DataFrame donde cada fila es un registro y cada columna es un atributo.

    Retorna
    -------
    list[list]
        Lista de transacciones, una por fila del DataFrame.
    """
    transacciones = []

    for _, fila in df_items.iterrows():
        transaccion = set()

        for columna, valor in fila.items():
            # Ignoramos valores nulos, NaN o cadenas vacías, ya que no
            # aportan información y ensuciarían las reglas generadas.
            if pd.notna(valor) and str(valor).strip() not in ("", "nan"):
                item = f"{columna}={str(valor).strip()}"
                transaccion.add(item)

        transacciones.append(frozenset(transaccion))

    return transacciones

In [6]:
transacciones = crear_transacciones(data_apriori)

print(f"Número de transacciones: {len(transacciones)}")
print("\nEjemplo de transacción (índice 45):")
transacciones[45]

Número de transacciones: 84738

Ejemplo de transacción (índice 45):


frozenset({'ENTIDAD=TAMAULIPAS',
           'GRUPO_EDAD=18-29',
           'MES_DESAPARICION=09',
           'MUNICIPIO=REYNOSA',
           'SEXENIO_DESAPARICION=Sheinbaum (2024-2030)',
           'SEXENIO_REGISTRO=Sheinbaum (2024-2030)',
           'SEXO=HOMBRE'})

## **Implementación de Apriori**

Apriori es un algoritmo clásico para encontrar **itemsets frecuentes**, es decir,
conjuntos de ítems que aparecen juntos con una frecuencia mínima en las transacciones.
Su nombre viene del principio *a priori*: **todo subconjunto de un itemset frecuente
también debe ser frecuente**. Esto permite podar el espacio de búsqueda y evitar
evaluar candidatos que de antemano sabemos que no cumplirán el umbral mínimo de soporte.

### Soporte

El soporte mide qué tan frecuente es un itemset en el dataset:

$$\text{soporte}(X) = \frac{\text{transacciones que contienen } X}{\text{total de transacciones}}$$

Solo se conservan los itemsets cuyo soporte sea mayor o igual al **soporte mínimo**
(`min_support`), que es un parámetro definido por el usuario.

### Pasos del algoritmo

1. Calcular el soporte de todos los ítems individuales y conservar los **frecuentes** (tamaño 1).
2. Generar **candidatos** de tamaño $k+1$ combinando los itemsets frecuentes de tamaño $k$.
3. Calcular el soporte de cada candidato.
4. Conservar únicamente los candidatos que superen el soporte mínimo.
5. Repetir los pasos 2–4 incrementando $k$ en cada iteración.
6. Detenerse cuando no existan nuevos candidatos frecuentes.

### **Funciones auxiliares**

In [7]:
def calcular_soporte_absoluto(itemset, transacciones):
    """
    Calcula el soporte absoluto de un itemset.

    Recorre todas las transacciones y cuenta en cuántas de ellas
    el itemset está contenido como subconjunto.

    Parámetros
    ----------
    itemset : frozenset
        Conjunto de ítems cuyo soporte se desea calcular.
    transacciones : list[frozenset]
        Lista de transacciones del dataset.

    Retorna
    -------
    int
        Número de transacciones que contienen al itemset.
    """
    return sum(1 for transaccion in transacciones if itemset.issubset(transaccion))


def obtener_itemsets_frecuentes_1(transacciones, min_soporte_absoluto):
    """
    Obtiene los itemsets frecuentes de tamaño 1 (F1).

    Recorre todas las transacciones contando las ocurrencias de cada
    ítem individual. Solo se conservan los ítems cuyo conteo supere
    el soporte mínimo absoluto.

    Parámetros
    ----------
    transacciones : list[frozenset]
        Lista de transacciones del dataset.
    min_soporte_absoluto : int
        Número mínimo de transacciones en las que debe aparecer un ítem
        para considerarse frecuente.

    Retorna
    -------
    dict[frozenset, int]
        Diccionario que mapea cada itemset frecuente de tamaño 1
        a su conteo absoluto.
    """
    conteos = defaultdict(int)
    for transaccion in transacciones:
        for item in transaccion:
            conteos[frozenset([item])] += 1

    return {
        itemset: conteo
        for itemset, conteo in conteos.items()
        if conteo >= min_soporte_absoluto
    }


def tiene_subconjuntos_frecuentes(candidato, frecuentes_anteriores):
    """
    Verifica la propiedad Apriori (poda por subconjuntos).

    Un candidato de tamaño k solo puede ser frecuente si **todos** sus
    subconjuntos de tamaño k-1 también lo son. Si alguno no aparece en
    los frecuentes anteriores, el candidato se descarta de inmediato,
    ahorrando el cálculo de su soporte.

    Parámetros
    ----------
    candidato : frozenset
        Itemset candidato de tamaño k.
    frecuentes_anteriores : iterable[frozenset]
        Itemsets frecuentes de tamaño k-1.

    Retorna
    -------
    bool
        True si todos los subconjuntos de tamaño k-1 son frecuentes,
        False en cuanto se encuentre uno que no lo sea.
    """
    k = len(candidato)
    frecuentes_anteriores = set(frecuentes_anteriores)
    for subconjunto in combinations(candidato, k - 1):
        if frozenset(subconjunto) not in frecuentes_anteriores:
            return False
    return True


def generar_candidatos(frecuentes_anteriores, k):
    """
    Genera candidatos de tamaño k a partir de los itemsets frecuentes de tamaño k-1.

    Se combinan pares de itemsets frecuentes mediante unión de conjuntos.
    Solo se conservan los candidatos que:
      1. Tienen exactamente k ítems.
      2. Pasan la poda Apriori (todos sus subconjuntos de tamaño k-1 son frecuentes).

    Parámetros
    ----------
    frecuentes_anteriores : iterable[frozenset]
        Itemsets frecuentes de tamaño k-1.
    k : int
        Tamaño deseado para los candidatos generados.

    Retorna
    -------
    set[frozenset]
        Conjunto de candidatos que superaron la poda Apriori.
    """
    candidatos = set()
    frecuentes_anteriores = list(frecuentes_anteriores)

    for i in range(len(frecuentes_anteriores)):
        for j in range(i + 1, len(frecuentes_anteriores)):
            candidato = frecuentes_anteriores[i] | frecuentes_anteriores[j]
            if len(candidato) == k:
                if tiene_subconjuntos_frecuentes(candidato, frecuentes_anteriores):
                    candidatos.add(candidato)

    return candidatos


def filtrar_candidatos_por_soporte(candidatos, transacciones, min_soporte_absoluto):
    """
    Filtra los candidatos que superan el soporte mínimo absoluto.

    Para cada candidato se calcula su soporte absoluto recorriendo
    las transacciones. Solo se conservan los que cumplen el umbral.

    Parámetros
    ----------
    candidatos : set[frozenset]
        Candidatos a evaluar.
    transacciones : list[frozenset]
        Lista de transacciones del dataset.
    min_soporte_absoluto : int
        Umbral mínimo de soporte absoluto.

    Retorna
    -------
    dict[frozenset, int]
        Diccionario con los candidatos frecuentes y su conteo absoluto.
    """
    return {
        candidato: conteo
        for candidato in candidatos
        if (conteo := calcular_soporte_absoluto(candidato, transacciones))
        >= min_soporte_absoluto
    }

### **Implementación algoritmo**

In [8]:
def apriori(transacciones, min_support=0.02, max_k=3, mostrar_proceso=True):
    """
    Implementación del algoritmo Apriori para minería de itemsets frecuentes.

    Encuentra todos los itemsets cuyo soporte relativo supera el umbral
    `min_support`, hasta un tamaño máximo de `max_k` ítems por itemset.

    Parámetros
    ----------
    transacciones : list[frozenset]
        Lista de transacciones del dataset.
    min_support : float o int
        Soporte mínimo. Si está entre 0 y 1 se interpreta como fracción
        relativa (por ejemplo 0.02 = 2%). Si es >= 1 se interpreta como
        conteo absoluto.
    max_k : int o None
        Tamaño máximo de itemset a generar. Si es None, no hay límite.
    mostrar_proceso : bool
        Si es True, imprime el progreso de cada nivel k.

    Retorna
    -------
    soportes : dict[frozenset, float]
        Soporte relativo de cada itemset frecuente.
    conteos : dict[frozenset, int]
        Soporte absoluto de cada itemset frecuente.
    frecuentes_por_nivel : dict[int, dict[frozenset, int]]
        Itemsets frecuentes agrupados por tamaño k.
    """
    n = len(transacciones)
    min_soporte_absoluto = ceil(min_support * n) if 0 < min_support < 1 else int(min_support)

    soportes              = {}
    conteos               = {}
    frecuentes_por_nivel  = {}

    if mostrar_proceso:
        print(f"Número de transacciones:  {n}")
        print(f"Soporte mínimo absoluto:  {min_soporte_absoluto}")
        print(f"Soporte mínimo relativo:  {min_soporte_absoluto/n:.4f}")
        print()

    # ── Nivel 1: itemsets frecuentes de tamaño 1 ─────────────────────────────
    k  = 1
    Fk = obtener_itemsets_frecuentes_1(transacciones, min_soporte_absoluto)
    frecuentes_por_nivel[k] = Fk

    for itemset, conteo in Fk.items():
        conteos[itemset]   = conteo
        soportes[itemset]  = conteo / n

    if mostrar_proceso:
        print(f"F{k}: {len(Fk)} itemsets frecuentes")

    # ── Niveles k > 1: generar candidatos, podar y filtrar ───────────────────
    while Fk:
        k += 1
        if max_k is not None and k > max_k:
            break

        Ck = generar_candidatos(Fk.keys(), k)
        if mostrar_proceso:
            print(f"C{k}: {len(Ck)} candidatos generados")

        if not Ck:
            break

        Fk = filtrar_candidatos_por_soporte(Ck, transacciones, min_soporte_absoluto)
        if mostrar_proceso:
            print(f"F{k}: {len(Fk)} itemsets frecuentes")

        if not Fk:
            break

        frecuentes_por_nivel[k] = Fk
        for itemset, conteo in Fk.items():
            conteos[itemset]  = conteo
            soportes[itemset] = conteo / n

    return soportes, conteos, frecuentes_por_nivel

### **Ejecución de apriori**

In [9]:
MIN_SUPPORT = 0.02
MAX_K = 3

import time
inicio = time.time()

soportes, conteos, frecuentes_por_nivel = apriori(
    transacciones,
    min_support=MIN_SUPPORT,
    max_k=MAX_K,
    mostrar_proceso=True
)

tiempo_propio = time.time() - inicio
print(f"\nTiempo de ejecución (Apriori propio): {tiempo_propio:.2f} segundos")

print("Total de itemsets frecuentes encontrados:", len(soportes))

print("\nItemsets frecuentes por tamaño:")
for k, itemsets in frecuentes_por_nivel.items():
    print(f"Tamaño {k}: {len(itemsets)}")

Número de transacciones:  84738
Soporte mínimo absoluto:  1695
Soporte mínimo relativo:  0.0200

F1: 51 itemsets frecuentes
C2: 1275 candidatos generados
F2: 149 itemsets frecuentes
C3: 180 candidatos generados
F3: 86 itemsets frecuentes

Tiempo de ejecución (Apriori propio): 11.29 segundos
Total de itemsets frecuentes encontrados: 286

Itemsets frecuentes por tamaño:
Tamaño 1: 51
Tamaño 2: 149
Tamaño 3: 86


### **Observaciones sobre los itemsets frecuentes encontrados**

Con un soporte mínimo del 2% (al menos 1,695 transacciones), el algoritmo encontró
**286 itemsets frecuentes** en total. Algunos puntos a destacar:

- **F1 (51 itemsets):** Hay 51 ítems individuales que aparecen en al menos el 2% de
  las transacciones. Esto indica que el dataset tiene una diversidad moderada de valores
  frecuentes.

- **F2 (149 itemsets):** De los 1,275 candidatos de tamaño 2 generados, solo 149
  superaron el umbral. La poda Apriori eliminó el 88% de los candidatos antes de
  calcular su soporte, lo que demuestra la eficiencia del principio *a priori*.

- **F3 (86 itemsets):** El decrecimiento de F2 a F3 es el comportamiento esperado
  en Apriori: conforme aumenta el tamaño del itemset, es más difícil superar el
  umbral mínimo de soporte, ya que se requiere que una combinación más específica
  de ítems aparezca juntos con suficiente frecuencia.

In [10]:
def itemset_a_texto(itemset):
    # Convierte un itemset en texto para mostrarlo en tablas.
    return ", ".join(sorted(itemset))

itemsets_frecuentes = []

for itemset, soporte in soportes.items():
    itemsets_frecuentes.append({
        "itemset": itemset_a_texto(itemset),
        "tamaño": len(itemset),
        "conteo_soporte": conteos[itemset],
        "soporte": soporte
    })

df_itemsets_frecuentes = pd.DataFrame(itemsets_frecuentes)

df_itemsets_frecuentes = df_itemsets_frecuentes.sort_values(
    by=["tamaño", "soporte"],
    ascending=[True, False]
)

df_itemsets_frecuentes

,itemset,tamaño,conteo_soporte,soporte
2,SEXO=HOMBRE,1,64666,0.763129
12,SEXENIO_REGISTRO=AMLO (2018-2024),1,29181,0.344367
11,SEXENIO_DESAPARICION=AMLO (2018-2024),1,26265,0.309955
25,SEXENIO_DESAPARICION=Peña Nieto (2012-2018),1,22927,0.270563
26,SEXENIO_REGISTRO=Peña Nieto (2012-2018),1,22533,0.265914
...,...,...,...,...
267,"MES_DESAPARICION=04, SEXENIO_DESAPARICION=AMLO (2018-2024), SEXO=HOMBRE",3,1724,0.020345
268,"MES_DESAPARICION=08, SEXENIO_DESAPARICION=AMLO (2018-2024), SEXO=HOMBRE",3,1722,0.020321
258,"ENTIDAD=SE DESCONOCE, MUNICIPIO=SE DESCONOCE, SEXO=HOMBRE",3,1719,0.020286
221,"ENTIDAD=SONORA, SEXENIO_REGISTRO=AMLO (2018-2024), SEXO=HOMBRE",3,1707,0.020144


# **Ejercicio 2:** Generación de reglas de asociación

## **Preparación de datos para mlxtend**

Los algoritmos de `mlxtend` (`Apriori` y `FP-Growth`) esperan los datos en un formato
distinto al que usamos en nuestra implementación. En lugar de una lista de transacciones,
requieren un **DataFrame one-hot encoded**, donde:

- Cada **fila** representa una transacción.
- Cada **columna** representa un ítem posible del dataset.
- Los **valores** son booleanos (`True`/`False`) que indican si el ítem
  está presente en esa transacción.

### ¿Por qué este formato?

`mlxtend` está optimizado para operar sobre matrices densas o dispersas de valores
booleanos, lo que le permite calcular el soporte de múltiples ítems simultáneamente
mediante operaciones vectorizadas de `numpy`. Esto es considerablemente más eficiente
que recorrer cada transacción una por una, como lo hace nuestra implementación.

### Transformación necesaria

Usaremos `TransactionEncoder` de `mlxtend`, que automatiza este proceso:

1. Aprende el vocabulario completo de ítems presentes en las transacciones.
2. Transforma cada transacción en un vector booleano de longitud fija.
3. Devuelve un array que convertimos en un DataFrame de pandas.

In [11]:
from mlxtend.preprocessing import TransactionEncoder

# TransactionEncoder aprende el vocabulario completo de ítems
# y transforma cada transacción en un vector booleano.
te = TransactionEncoder()

# fit() aprende todos los ítems únicos presentes en las transacciones.
# transform() convierte cada transacción en un vector booleano.
te_array = te.fit(transacciones).transform(transacciones)

# Convertimos el array a DataFrame usando los nombres de los ítems como columnas.
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f"Dimensiones del DataFrame encoded: {df_encoded.shape}")
print(f"Ítems únicos (columnas): {df_encoded.shape[1]}")
df_encoded.head()

Dimensiones del DataFrame encoded: (84738, 1619)
Ítems únicos (columnas): 1619


,ENTIDAD=AGUASCALIENTES,ENTIDAD=BAJA CALIFORNIA,ENTIDAD=BAJA CALIFORNIA SUR,ENTIDAD=CAMPECHE,ENTIDAD=CHIAPAS,ENTIDAD=CHIHUAHUA,ENTIDAD=CIUDAD DE MÉXICO,ENTIDAD=COAHUILA,ENTIDAD=COLIMA,ENTIDAD=DURANGO,...,SEXENIO_DESAPARICION=Zedillo (1994-2000),SEXENIO_REGISTRO=AMLO (2018-2024),SEXENIO_REGISTRO=Calderón (2006-2012),SEXENIO_REGISTRO=Fox (2000-2006),SEXENIO_REGISTRO=Peña Nieto (2012-2018),SEXENIO_REGISTRO=Sheinbaum (2024-2030),SEXENIO_REGISTRO=Zedillo (1994-2000),SEXO=HOMBRE,SEXO=INDETERMINADO,SEXO=MUJER
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,True
3,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,False


## **Evaluación de modelos mlxtend**

### **Apriori de mlxtend**

In [12]:
from mlxtend.frequent_patterns import apriori as apriori_mlxtend
import time

inicio = time.time()

frecuentes_mlxtend = apriori_mlxtend(
    df_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_K
)

tiempo_mlxtend = time.time() - inicio
print(f"Tiempo de ejecución (Apriori mlxtend): {tiempo_mlxtend:.2f} segundos")
print(f"Itemsets frecuentes encontrados: {len(frecuentes_mlxtend)}")
frecuentes_mlxtend.head()

Tiempo de ejecución (Apriori mlxtend): 0.44 segundos
Itemsets frecuentes encontrados: 286


,support,itemsets
0,0.035462,frozenset({ENTIDAD=BAJA CALIFORNIA})
1,0.044384,frozenset({ENTIDAD=CHIHUAHUA})
2,0.033385,frozenset({ENTIDAD=CIUDAD DE MÉXICO})
3,0.030506,frozenset({ENTIDAD=COAHUILA})
4,0.131027,frozenset({ENTIDAD=ESTADO DE MÉXICO})


### **FP-Growth**

In [13]:
from mlxtend.frequent_patterns import fpgrowth
import time

inicio = time.time()

frecuentes_fpgrowth = fpgrowth(
    df_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_K
)

tiempo_fpgrowth = time.time() - inicio
print(f"Tiempo de ejecución (FP-Growth): {tiempo_fpgrowth:.2f} segundos")
print(f"Itemsets frecuentes encontrados: {len(frecuentes_fpgrowth)}")
frecuentes_fpgrowth.head()

Tiempo de ejecución (FP-Growth): 3.01 segundos
Itemsets frecuentes encontrados: 286


,support,itemsets
0,0.763129,frozenset({SEXO=HOMBRE})
1,0.177240,frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)})
2,0.175907,frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030)})
3,0.080932,frozenset({GRUPO_EDAD=45-59})
4,0.080188,frozenset({MES_DESAPARICION=09})


## **Comparación de algoritmos**

A continuación se comparan los tres algoritmos ejecutados con los mismos parámetros:
`min_support=0.02` y `max_k=3`.

| Algoritmo | Tiempo de ejecución | Itemsets frecuentes |
|---|---|---|
| Apriori (implementación propia) | 11.85 s | 286 |
| Apriori (mlxtend) | 0.45 s | 286 |
| FP-Growth (mlxtend) | 2.97 s | 286 |

---

### ¿Cuál tarda más?

La **implementación propia de Apriori** fue la más lenta con **11.85 segundos**. Esto
se debe a varias razones:

- Está implementada en **Python puro**, sin aprovechar operaciones vectorizadas.
- La función `calcular_soporte_absoluto` recorre **todas las transacciones** una por una
  para cada candidato, lo que resulta en una complejidad computacional alta.
- El uso de `iterrows()` en la construcción de transacciones es inherentemente lento
  para datasets grandes.

---

### ¿Cuál tarda menos y por qué?

En este caso, **Apriori de mlxtend** fue el más rápido con **0.45 segundos**,
seguido de FP-Growth con **2.97 segundos**. Esto es llamativo porque en teoría
FP-Growth debería ser más eficiente, sin embargo, para este dataset en particular,
Apriori de mlxtend resultó más rápido. Esto puede deberse a que:

- Con un soporte mínimo del 2% y solo 286 itemsets frecuentes, el espacio de
  búsqueda es lo suficientemente pequeño como para que las operaciones vectorizadas
  de NumPy de mlxtend sean más rápidas que la construcción y minería del FP-Tree.
- La complejidad de construir el FP-Tree puede superar su beneficio cuando el
  número de itemsets frecuentes es reducido.

**Apriori de mlxtend** es rápido porque:

- Está implementado sobre **operaciones vectorizadas de NumPy**, que operan sobre
  matrices booleanas de forma altamente optimizada.
- Calcula el soporte de múltiples candidatos simultáneamente en lugar de evaluarlos
  uno por uno.

**FP-Growth** sigue siendo eficiente por su diseño, ya que:

- En lugar de generar candidatos explícitamente, construye un **FP-Tree** (árbol de
  patrones frecuentes), una estructura compacta que comprime el dataset en memoria.
- Evita el costoso proceso de **múltiples pasadas** sobre las transacciones para
  calcular el soporte de cada candidato.
- Extrae los itemsets frecuentes directamente del árbol sin necesidad de generar
  y podar candidatos.

---

### ¿Generan las mismas reglas de asociación? ¿Por qué?

Sí, los tres algoritmos encontraron exactamente **286 itemsets frecuentes**, lo que
confirma que producen resultados equivalentes. Esto era esperado porque los tres
resuelven el mismo problema. La diferencia entre ellos no está en qué encuentran, sino en cómo lo hacen.



## **Generación de reglas de asociación**

A partir de los itemsets frecuentes se generan **reglas de asociación**, que expresan
relaciones del tipo:

$$\text{antecedente} \Rightarrow \text{consecuente}$$

Por ejemplo:

$$\text{SEXO=MUJER, GRUPO\_EDAD=12-17} \Rightarrow \text{ENTIDAD=ESTADO DE MÉXICO}$$

Esta regla se leería como: *"cuando la víctima es mujer y tiene entre 12 y 17 años,
es frecuente que la desaparición haya ocurrido en el Estado de México"*.

### Medidas de interés

Para determinar si una regla es útil o trivial, se evalúan tres métricas:

**Soporte**
Mide qué tan frecuente es la regla en el dataset. Un soporte alto indica que la
combinación antecedente + consecuente aparece en una porción significativa de los casos.

$$\text{soporte}(X \Rightarrow Y) = \frac{|\text{transacciones que contienen } X \cup Y|}{|\text{total de transacciones}|}$$

**Confianza**
Mide la fiabilidad de la regla: dado que ocurre el antecedente, ¿qué tan probable
es que ocurra el consecuente?

$$\text{confianza}(X \Rightarrow Y) = \frac{\text{soporte}(X \cup Y)}{\text{soporte}(X)}$$

**Lift**
Mide si la asociación entre antecedente y consecuente es real o simplemente producto
de que ambos son frecuentes por separado.

$$\text{lift}(X \Rightarrow Y) = \frac{\text{confianza}(X \Rightarrow Y)}{\text{soporte}(Y)}$$

| Valor de lift | Interpretación |
|---|---|
| $> 1$ | Asociación positiva: el antecedente favorece al consecuente |
| $= 1$ | Independencia: no hay asociación real |
| $< 1$ | Asociación negativa: el antecedente inhibe al consecuente |

In [14]:
from mlxtend.frequent_patterns import association_rules
MIN_CONFIDENCE = 0.70

reglas_mlxtend = association_rules(
    frecuentes_mlxtend,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

print(f"Total de reglas encontradas (mlxtend): {len(reglas_mlxtend)}")
reglas_mlxtend.head(20)

Total de reglas encontradas (mlxtend): 196


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({MUNICIPIO=TIJUANA}),frozenset({ENTIDAD=BAJA CALIFORNIA}),0.024357,0.035462,0.024357,1.000000,28.199002,1.0,0.023494,inf,0.988618,0.686855,1.000000,0.843428
1,frozenset({ENTIDAD=BAJA CALIFORNIA}),frozenset({SEXO=HOMBRE}),0.035462,0.763129,0.026128,0.736772,0.965462,1.0,-0.000935,0.899871,-0.035762,0.033824,-0.111270,0.385505
2,frozenset({ENTIDAD=CHIHUAHUA}),frozenset({SEXO=HOMBRE}),0.044384,0.763129,0.038153,0.859612,1.126431,1.0,0.004282,1.687259,0.117453,0.049590,0.407323,0.454804
3,frozenset({ENTIDAD=COAHUILA}),frozenset({SEXO=HOMBRE}),0.030506,0.763129,0.023649,0.775242,1.015873,1.0,0.000370,1.053894,0.016117,0.030714,0.051138,0.403116
4,frozenset({MUNICIPIO=ATLAUTLA}),frozenset({ENTIDAD=ESTADO DE MÉXICO}),0.021655,0.131027,0.021655,1.000000,7.631991,1.0,0.018818,inf,0.888207,0.165271,1.000000,0.582635
5,frozenset({ENTIDAD=GUANAJUATO}),frozenset({SEXO=HOMBRE}),0.025809,0.763129,0.021773,0.843621,1.105477,1.0,0.002077,1.514730,0.097941,0.028381,0.339816,0.436076
6,frozenset({ENTIDAD=GUERRERO}),frozenset({SEXO=HOMBRE}),0.047381,0.763129,0.039486,0.833375,1.092050,1.0,0.003328,1.421582,0.088484,0.051213,0.296558,0.442559
7,frozenset({ENTIDAD=JALISCO}),frozenset({SEXO=HOMBRE}),0.056775,0.763129,0.048868,0.860736,1.127904,1.0,0.005542,1.700877,0.120225,0.063380,0.412068,0.462386
8,frozenset({ENTIDAD=MICHOACÁN}),frozenset({SEXO=HOMBRE}),0.069485,0.763129,0.058026,0.835088,1.094296,1.0,0.005000,1.436352,0.092605,0.074912,0.303792,0.455563
9,frozenset({ENTIDAD=NUEVO LEÓN}),frozenset({SEXO=HOMBRE}),0.043275,0.763129,0.032016,0.739842,0.969485,1.0,-0.001008,0.910490,-0.031851,0.041344,-0.098310,0.390898


## **Filtrado de reglas**

De las **196 reglas** generadas, es necesario aplicar filtros para identificar
aquellas que son realmente útiles e interesantes. A continuación se describen
los criterios de filtrado aplicados.

In [15]:
def filtrar_reglas(df_reglas, contiene_antecedente=None, contiene_consecuente=None,
                   min_lift=1.0, min_confidence=None, min_support=None):
    """
    Filtra reglas de asociación según criterios especificados.

    Parámetros
    ----------
    df_reglas : pd.DataFrame
        DataFrame de reglas generado por mlxtend.
    contiene_antecedente : str, list[str] o None
        Filtra reglas cuyo antecedente contenga todos estos strings.
    contiene_consecuente : str, list[str] o None
        Filtra reglas cuyo consecuente contenga todos estos strings.
    min_lift : float
        Lift mínimo (default 1.0 para descartar asociaciones negativas).
    min_confidence : float o None
        Confianza mínima adicional (si None, usa la del DataFrame).
    min_support : float o None
        Soporte mínimo adicional (si None, usa la del DataFrame).

    Retorna
    -------
    pd.DataFrame
        Subconjunto de reglas que cumplen todos los criterios,
        ordenado por lift descendente.
    """
    mask = pd.Series([True] * len(df_reglas), index=df_reglas.index)

    if contiene_antecedente:
        if isinstance(contiene_antecedente, str):
            contiene_antecedente = [contiene_antecedente]
        for item in contiene_antecedente:
            mask &= df_reglas["antecedents"].apply(
                lambda x, i=item: any(i in elem for elem in x)
            )

    if contiene_consecuente:
        if isinstance(contiene_consecuente, str):
            contiene_consecuente = [contiene_consecuente]
        for item in contiene_consecuente:
            mask &= df_reglas["consequents"].apply(
                lambda x, i=item: any(i in elem for elem in x)
            )

    if min_lift is not None:
        mask &= df_reglas["lift"] >= min_lift

    if min_confidence is not None:
        mask &= df_reglas["confidence"] >= min_confidence

    if min_support is not None:
        mask &= df_reglas["support"] >= min_support

    return (
        df_reglas[mask][["antecedents", "consequents", "support", "confidence", "lift"]]
        .sort_values("lift", ascending=False)
        .reset_index(drop=True)
    )

### Eliminación de reglas con lift similar a 1.0
Algunas reglas tienen lift muy cercano a 1, lo que indica una asociación prácticamente nula. Necesitamos un umbral de lift más exigente.

In [16]:
reglas = filtrar_reglas(
    reglas_mlxtend,
    min_lift=1.2
)
print(f"Reglas tras filtrado: {len(reglas)}")
reglas

Reglas tras filtrado: 120


,antecedents,consequents,support,confidence,lift
0,frozenset({MUNICIPIO=TIJUANA}),frozenset({ENTIDAD=BAJA CALIFORNIA}),0.024357,1.000000,28.199002
1,frozenset({MUNICIPIO=CULIACÁN}),frozenset({ENTIDAD=SINALOA}),0.020097,1.000000,17.991083
2,frozenset({ENTIDAD=SE DESCONOCE}),"frozenset({MUNICIPIO=SE DESCONOCE, SEXO=HOMBRE})",0.020286,0.725623,16.301116
3,frozenset({ENTIDAD=SE DESCONOCE}),frozenset({MUNICIPIO=SE DESCONOCE}),0.027957,1.000000,15.703855
4,"frozenset({ENTIDAD=SE DESCONOCE, SEXO=HOMBRE})",frozenset({MUNICIPIO=SE DESCONOCE}),0.020286,1.000000,15.703855
...,...,...,...,...,...
115,"frozenset({SEXO=MUJER, SEXENIO_REGISTRO=AMLO (2018-2024)})",frozenset({SEXENIO_DESAPARICION=AMLO (2018-2024)}),0.067243,0.773449,2.495356
116,frozenset({ENTIDAD=TABASCO}),"frozenset({SEXENIO_DESAPARICION=AMLO (2018-2024), SEXENIO_REGISTRO=AMLO (2018-2024)})",0.021985,0.709985,2.433569
117,"frozenset({ENTIDAD=MICHOACÁN, SEXENIO_DESAPARICION=AMLO (2018-2024)})",frozenset({SEXENIO_REGISTRO=AMLO (2018-2024)}),0.028134,0.805950,2.340379
118,frozenset({ENTIDAD=TABASCO}),frozenset({SEXENIO_DESAPARICION=AMLO (2018-2024)}),0.022009,0.710747,2.293062


### Eliminación de reglas geográficas triviales

Las reglas cuyo consecuente es `ENTIDAD` son trivialmente ciertas por definición
geográfica: un municipio siempre pertenece a la misma entidad federativa. Por ejemplo:

$$\text{MUNICIPIO=TIJUANA} \Rightarrow \text{ENTIDAD=BAJA CALIFORNIA}$$

Estas reglas tienen confianza = 1.0 y lift extremadamente alto, pero no aportan
ningún conocimiento nuevo sobre el fenómeno de desapariciones. Su presencia
distorsiona el ranking de reglas y dificulta identificar patrones realmente
interesantes. Por esta razón, excluimos aquellas reglas donde el consecuente
contenga `ENTIDAD`.

In [17]:
reglas_limpias = reglas

reglas_limpias = reglas_limpias[
    ~reglas_limpias["consequents"].apply(
        lambda x: any("ENTIDAD" in item for item in x)
    ) &
    ~reglas_limpias["antecedents"].apply(
        lambda x: any("SE DESCONOCE" in item for item in x)
    ) &
    ~reglas_limpias["consequents"].apply(
        lambda x: any("SE DESCONOCE" in item for item in x)
    )
].reset_index(drop=True)

print(f"Reglas tras filtrado: {len(reglas_limpias)}")
reglas_limpias

Reglas tras filtrado: 113


,antecedents,consequents,support,confidence,lift
0,"frozenset({SEXENIO_REGISTRO=Calderón (2006-2012), ENTIDAD=TAMAULIPAS})",frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),0.027461,1.000000,7.718892
1,"frozenset({GRUPO_EDAD=30-44, SEXENIO_REGISTRO=Calderón (2006-2012)})",frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),0.025408,0.999536,7.715309
2,"frozenset({SEXENIO_REGISTRO=Calderón (2006-2012), GRUPO_EDAD=18-29})",frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),0.022741,0.999481,7.714889
3,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), GRUPO_EDAD=18-29})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.022741,0.986182,7.668818
4,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), SEXO=HOMBRE})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.103342,0.984929,7.659069
...,...,...,...,...,...
108,"frozenset({SEXO=MUJER, SEXENIO_REGISTRO=AMLO (2018-2024)})",frozenset({SEXENIO_DESAPARICION=AMLO (2018-2024)}),0.067243,0.773449,2.495356
109,frozenset({ENTIDAD=TABASCO}),"frozenset({SEXENIO_DESAPARICION=AMLO (2018-2024), SEXENIO_REGISTRO=AMLO (2018-2024)})",0.021985,0.709985,2.433569
110,"frozenset({ENTIDAD=MICHOACÁN, SEXENIO_DESAPARICION=AMLO (2018-2024)})",frozenset({SEXENIO_REGISTRO=AMLO (2018-2024)}),0.028134,0.805950,2.340379
111,frozenset({ENTIDAD=TABASCO}),frozenset({SEXENIO_DESAPARICION=AMLO (2018-2024)}),0.022009,0.710747,2.293062


## **Exploración de reglas de asociación**

Con las 113 reglas filtradas (lift ≥ 1.2, sin reglas geográficas triviales ni valores
desconocidos), procedemos a explorar patrones relevantes agrupados por temática.
El objetivo es identificar asociaciones no obvias que aporten información real sobre
el fenómeno de desapariciones en México.

Propusimos los siguientes patrones a explorar:

1. **Sexenio de desaparición → Sexenio de registro**: ¿Existe una brecha temporal
   entre cuándo ocurre la desaparición y cuándo se registra? ¿Hay sexenios donde
   los casos se reportan con mayor rezago?

2. **Entidad → Sexenio**: ¿Hay entidades federativas cuyas desapariciones se concentran
   en un período de gobierno específico? ¿Refleja esto un fenómeno real o un sesgo
   en el sistema de registro?

3. **Sexo → Entidad o Sexenio**: ¿Existen diferencias en los patrones de desaparición
   entre hombres y mujeres según la región o el período de gobierno?

4. **Grupo de edad → Sexenio o Entidad**: ¿Ciertos grupos de edad están más asociados
   a determinados períodos o regiones del país?

### Patrón 1: Sexenio de desaparición → Sexenio de registro
Con este patrón queremos explorar si existen brechas temporales entre desapariciones y registros entre sexenios 

In [26]:
# Patrón 1: ¿El sexenio en que ocurre la desaparición predice el sexenio de registro?
patron_1 = filtrar_reglas(
    reglas_mlxtend,
    contiene_antecedente="SEXENIO_DESAPARICION",
    contiene_consecuente="SEXENIO_REGISTRO",
    min_lift=1.2
)
print(f"Cantidad de reglas: {len(patron_1)}")
patron_1.head(50)

Cantidad de reglas: 56


,antecedents,consequents,support,confidence,lift
0,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), GRUPO_EDAD=18-29})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.022741,0.986182,7.668818
1,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), SEXO=HOMBRE})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.103342,0.984929,7.659069
2,frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.127487,0.984059,7.652307
3,frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),"frozenset({SEXENIO_REGISTRO=Calderón (2006-2012), SEXO=HOMBRE})",0.103342,0.797686,7.651612
4,"frozenset({GRUPO_EDAD=30-44, SEXENIO_DESAPARICION=Calderón (2006-2012)})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.025408,0.981313,7.630951
5,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), SEXO=MUJER})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.024027,0.980732,7.626437
6,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), ENTIDAD=TAMAULIPAS})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.027461,0.960380,7.468171
7,"frozenset({ENTIDAD=ESTADO DE MÉXICO, SEXENIO_DESAPARICION=Sheinbaum (2024-2030)})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.036725,0.999679,5.640241
8,"frozenset({SEXO=MUJER, SEXENIO_DESAPARICION=Sheinbaum (2024-2030)})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.045293,0.998959,5.636179
9,"frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030), GRUPO_EDAD=18-29})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.052951,0.998887,5.635773


El hallazgo principal es que: **las desapariciones tienden a registrarse dentro del mismo
sexenio en que ocurren**, con confianzas cercanas al 100% y lifts muy elevados.

| Sexenio | Lift aproximado | Interpretación |
|---|---|---|
| Calderón (2006-2012) | ~7.7 | Asociación muy fuerte |
| Sheinbaum (2024-2030) | ~5.6 | Asociación muy fuerte |
| Peña Nieto (2012-2018) | ~3.6 | Asociación fuerte |
| AMLO (2018-2024) | ~2.8 | Asociación moderada |

Algunos puntos a destacar:

- **El sexenio de Calderón presenta el lift más alto (7.7)**, lo que indica que las
  desapariciones ocurridas en ese período se registraron de forma casi exclusiva
  durante el mismo sexenio. Esto coincide históricamente con el inicio de la guerra
  contra el narcotráfico, período en que el fenómeno de desapariciones cobró mayor
  visibilidad institucional.

- **El sexenio de Sheinbaum tiene un lift alto (5.6)** a pesar de ser el más reciente,
  lo cual es esperable: los casos recientes naturalmente se registran en el mismo
  período en que ocurren, ya que no ha habido tiempo para un rezago significativo.

- **El sexenio de AMLO presenta el lift más bajo (2.8)**, lo que podría sugerir que
  una mayor proporción de casos ocurridos en ese período fueron registrados en sexenios
  distintos, ya sea por rezago administrativo o por la incorporación retroactiva de
  casos históricos al sistema de registro.

- **Una limitación importante**: estas reglas no nos dicen si las desapariciones
  *realmente* ocurrieron en ese sexenio, sino cuándo fueron *registradas*. El rezago
  entre la desaparición y el registro es en sí mismo un fenómeno que estas reglas
  ayudan a visibilizar.

### Patrón 2: Entidad → Sexenio
Este patrón explora si ciertas entidades federativas concentran sus desapariciones
en períodos de gobierno específicos

In [19]:
# Patrón 2: Entidad → Sexenio
filtrar_reglas(
    reglas_mlxtend,
    contiene_antecedente="ENTIDAD",
    contiene_consecuente="SEXENIO",
    min_lift=1.2
)

,antecedents,consequents,support,confidence,lift
0,"frozenset({SEXENIO_REGISTRO=Calderón (2006-2012), ENTIDAD=TAMAULIPAS})",frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),0.027461,1.000000,7.718892
1,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), ENTIDAD=TAMAULIPAS})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.027461,0.960380,7.468171
2,"frozenset({ENTIDAD=ESTADO DE MÉXICO, SEXENIO_REGISTRO=Sheinbaum (2024-2030)})",frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030)}),0.036725,0.997436,5.670248
3,"frozenset({ENTIDAD=ESTADO DE MÉXICO, SEXENIO_DESAPARICION=Sheinbaum (2024-2030)})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.036725,0.999679,5.640241
4,"frozenset({ENTIDAD=JALISCO, SEXENIO_DESAPARICION=Peña Nieto (2012-2018)})",frozenset({SEXENIO_REGISTRO=Peña Nieto (2012-2018)}),0.030600,0.998460,3.754826
5,"frozenset({ENTIDAD=ESTADO DE MÉXICO, SEXENIO_DESAPARICION=Peña Nieto (2012-2018)})",frozenset({SEXENIO_REGISTRO=Peña Nieto (2012-2018)}),0.026600,0.998229,3.753956
6,"frozenset({ENTIDAD=JALISCO, SEXENIO_REGISTRO=Peña Nieto (2012-2018)})",frozenset({SEXENIO_DESAPARICION=Peña Nieto (2012-2018)}),0.030600,0.999614,3.694567
7,"frozenset({SEXENIO_REGISTRO=Peña Nieto (2012-2018), ENTIDAD=TAMAULIPAS})",frozenset({SEXENIO_DESAPARICION=Peña Nieto (2012-2018)}),0.033338,0.997528,3.686856
8,"frozenset({ENTIDAD=ESTADO DE MÉXICO, SEXENIO_REGISTRO=Peña Nieto (2012-2018)})",frozenset({SEXENIO_DESAPARICION=Peña Nieto (2012-2018)}),0.026600,0.912920,3.374145
9,"frozenset({ENTIDAD=TABASCO, SEXENIO_REGISTRO=AMLO (2018-2024)})",frozenset({SEXENIO_DESAPARICION=AMLO (2018-2024)}),0.021985,0.996790,3.215913


Los resultados revelan asociaciones geográficos temporales relevantes:

**Tamaulipas → Sexenio de Calderón (lift ~7.7)**
Es la asociación más fuerte de este patrón. Esto es consistente con el contexto histórico: 
Tamaulipas fue uno de los estados más afectados por la violencia del crimen organizado durante la guerra
contra el narcotráfico, lo que se tradujo en un volumen elevado de registros en ese
sexenio.

**Estado de México → Sexenio de Sheinbaum (lift ~5.7) y Peña Nieto (lift ~3.75)**
El Estado de México aparece asociado tanto al sexenio de Sheinbaum como al de Peña
Nieto. La asociación con Peña Nieto es esperable dado que fue gobernador del Estado
de México antes de la presidencia. La asociación con Sheinbaum podría reflejar un
aumento en el registro de casos recientes en una de las entidades más pobladas del
país.

**Jalisco → Sexenio de Peña Nieto (lift ~3.75)**
Jalisco muestra una fuerte concentración de desapariciones registradas durante el
sexenio de Peña Nieto, período que coincide con la expansión del Cártel Jalisco
Nueva Generación como actor dominante en la región.

**Michoacán → Sexenio de AMLO (lift ~3.18)**
Michoacán aparece asociado al sexenio de AMLO, lo que puede reflejar tanto un
aumento real en las desapariciones durante ese período o una mejora en los
mecanismos de registro en el estado.

**Tabasco → Sexenio de AMLO (lift ~2.07-2.43)**
Tabasco presenta una asociación moderada pero consistente con el sexenio de AMLO.
Dado que AMLO es originario de Tabasco y tuvo una fuerte presencia política en el
estado, esta asociación podría reflejar tanto un fenómeno real como una mejora en
la capacidad institucional de registro durante ese período.

**Limitación importante**
Es fundamental recordar que estas reglas reflejan patrones en el *registro* de
casos, no necesariamente en la ocurrencia real de las desapariciones. Una entidad
puede aparecer asociada a un sexenio simplemente porque fue en ese período cuando
se fortaleció su sistema de registro, y no porque las desapariciones hayan aumentado.

### Patrón 3: Sexo → Entidad o Sexenio
Este patrón explora si ciertas entidades federativas concentran sus desapariciones
en períodos de gobierno específicos

In [20]:
# Patrón 3: Sexo → Entidad o Sexenio
filtrar_reglas(
    reglas_mlxtend,
    contiene_antecedente="SEXO",
    min_lift=1.2
)

,antecedents,consequents,support,confidence,lift
0,"frozenset({ENTIDAD=SE DESCONOCE, SEXO=HOMBRE})",frozenset({MUNICIPIO=SE DESCONOCE}),0.020286,1.000000,15.703855
1,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), SEXO=HOMBRE})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.103342,0.984929,7.659069
2,"frozenset({SEXO=MUJER, SEXENIO_REGISTRO=Calderón (2006-2012)})",frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),0.024027,0.991719,7.654976
3,"frozenset({SEXENIO_REGISTRO=Calderón (2006-2012), SEXO=HOMBRE})",frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),0.103342,0.991284,7.651612
4,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), SEXO=MUJER})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.024027,0.980732,7.626437
5,"frozenset({SEXO=MUJER, SEXENIO_REGISTRO=Sheinbaum (2024-2030)})",frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030)}),0.045293,0.994558,5.653889
6,"frozenset({SEXO=MUJER, SEXENIO_DESAPARICION=Sheinbaum (2024-2030)})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.045293,0.998959,5.636179
7,"frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030), SEXO=HOMBRE})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.130201,0.998462,5.633373
8,"frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030), SEXO=HOMBRE})",frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030)}),0.130201,0.989862,5.627191
9,"frozenset({SEXO=HOMBRE, SEXENIO_REGISTRO=Peña Nieto (2012-2018)})",frozenset({SEXENIO_DESAPARICION=Peña Nieto (2012-2018)}),0.203108,0.981859,3.628941


In [21]:
# ¿Hay asociación directa entre SEXO y ENTIDAD?
filtrar_reglas(
    reglas_mlxtend,
    contiene_antecedente="SEXO",
    contiene_consecuente="ENTIDAD",
    min_lift=1.2
)

,antecedents,consequents,support,confidence,lift


In [22]:
# ¿Hay asociación directa entre SEXO y GRUPO_EDAD?
filtrar_reglas(
    reglas_mlxtend,
    contiene_antecedente="SEXO",
    contiene_consecuente="GRUPO_EDAD",
    min_lift=1.2
)

,antecedents,consequents,support,confidence,lift


Las reglas obtenidas con `SEXO` en el antecedente no revelan asociaciones directas
entre el sexo de la víctima y una entidad federativa o grupo de edad específico con
lift significativo. Lo que sí aparece son variaciones del patrón 1 segmentadas por
sexo, es decir, reglas del tipo:

$$\text{SEXO=HOMBRE, SEXENIO\_DESAPARICION=X} \Rightarrow \text{SEXENIO\_REGISTRO=X}$$

Esto nos dice que **el comportamiento de registro por sexenio es consistente tanto
para hombres como para mujeres**: en ambos casos, las desapariciones tienden a
registrarse dentro del mismo sexenio en que ocurren, sin diferencias significativas
entre sexos.

La ausencia de asociaciones directas entre `SEXO` y `ENTIDAD` o `GRUPO_EDAD` es
en sí misma un hallazgo relevante: **no hay una entidad federativa que concentre
desproporcionadamente las desapariciones de un sexo específico** con la frecuencia
suficiente para superar el umbral de soporte del 2%. Esto podría indicar que el
fenómeno de desapariciones afecta a hombres y mujeres de forma geográficamente
distribuida, aunque con distintas magnitudes absolutas.

**Limitación**: esto no significa que no existan diferencias por sexo en el fenómeno
real. El umbral de soporte mínimo del 2% puede estar ocultando patrones locales
importantes que simplemente no son lo suficientemente frecuentes a nivel nacional
para ser capturados por Apriori.

### Patrón 4: Grupo de edad → Sexenio o entidad

In [27]:
# Patrón 4: Grupo de edad → Sexenio o Entidad
filtrar_reglas(
    reglas_mlxtend,
    contiene_antecedente="GRUPO_EDAD",
    min_lift=1.2
)

,antecedents,consequents,support,confidence,lift
0,"frozenset({GRUPO_EDAD=30-44, SEXENIO_REGISTRO=Calderón (2006-2012)})",frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),0.025408,0.999536,7.715309
1,"frozenset({SEXENIO_REGISTRO=Calderón (2006-2012), GRUPO_EDAD=18-29})",frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012)}),0.022741,0.999481,7.714889
2,"frozenset({SEXENIO_DESAPARICION=Calderón (2006-2012), GRUPO_EDAD=18-29})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.022741,0.986182,7.668818
3,"frozenset({GRUPO_EDAD=30-44, SEXENIO_DESAPARICION=Calderón (2006-2012)})",frozenset({SEXENIO_REGISTRO=Calderón (2006-2012)}),0.025408,0.981313,7.630951
4,"frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030), GRUPO_EDAD=18-29})",frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030)}),0.052951,0.993358,5.647069
5,"frozenset({GRUPO_EDAD=45-59, SEXENIO_REGISTRO=Sheinbaum (2024-2030)})",frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030)}),0.021336,0.991772,5.638049
6,"frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030), GRUPO_EDAD=18-29})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.052951,0.998887,5.635773
7,"frozenset({GRUPO_EDAD=45-59, SEXENIO_DESAPARICION=Sheinbaum (2024-2030)})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.021336,0.998343,5.632707
8,"frozenset({GRUPO_EDAD=30-44, SEXENIO_DESAPARICION=Sheinbaum (2024-2030)})",frozenset({SEXENIO_REGISTRO=Sheinbaum (2024-2030)}),0.051181,0.997929,5.630369
9,"frozenset({GRUPO_EDAD=30-44, SEXENIO_REGISTRO=Sheinbaum (2024-2030)})",frozenset({SEXENIO_DESAPARICION=Sheinbaum (2024-2030)}),0.051181,0.990183,5.629015


Las reglas con `GRUPO_EDAD` en el antecedente confirman el patrón dominante del
análisis: las desapariciones se registran dentro del mismo sexenio en que ocurren,
y este comportamiento es **consistente en todos los grupos de edad**. Sin embargo,
hay matices interesantes:

**Grupos de edad presentes en las reglas**

Los grupos que aparecen con suficiente frecuencia para generar reglas son:
- **18-29 años**: el grupo más frecuente, presente en todos los sexenios.
- **30-44 años**: segundo grupo más frecuente, también presente en todos los sexenios.
- **45-59 años**: aparece únicamente en los sexenios de Sheinbaum y AMLO.

Notablemente, los grupos **0-11** y **12-17** no generan reglas con lift significativo,
lo que indica que las desapariciones de menores de edad no alcanzan el umbral mínimo
de soporte del 2% en combinación con ningún sexenio. Esto no significa que no existan,
sino que están distribuidas de forma más dispersa en el dataset.

**Comportamiento por sexenio**

El patrón de registro es consistente entre grupos de edad dentro de cada sexenio:

| Sexenio | Grupos de edad | Lift aproximado |
|---|---|---|
| Calderón (2006-2012) | 18-29, 30-44 | ~7.7 |
| Sheinbaum (2024-2030) | 18-29, 30-44, 45-59 | ~5.6 |
| Peña Nieto (2012-2018) | 18-29, 30-44 | ~3.7 |
| AMLO (2018-2024) | 18-29, 30-44, 45-59 | ~3.2 |

Los lifts son prácticamente idénticos entre grupos de edad dentro de un mismo sexenio,
lo que sugiere que **la probabilidad de que un caso se registre en el mismo sexenio
en que ocurre no depende del grupo de edad de la víctima**.

**Hallazgo relevante**

No se encontraron asociaciones directas entre `GRUPO_EDAD` y una entidad federativa
o sexo específico con lift ≥ 1.2. Esto indica que el fenómeno de desapariciones
no muestra concentraciones geográficas o de género por grupo de edad lo suficientemente
fuertes para ser capturadas a nivel nacional con el soporte mínimo establecido.

# **Ejercicio 3:** Algoritmos de clasificación

Queremos predecir el estatus de la víctima. Notemos que tras filtrar los valores confidenciales, el estatus toma
dos valores posibles:

- `DESAPARECIDA`: 94.27% de los registros
- `NO LOCALIZADA`: 5.73% de los registros

### Modelo de clasificación

Se utilizará un **Random Forest Classifier**, un modelo de ensamble basado en la
construcción de múltiples árboles de decisión sobre subconjuntos aleatorios del
dataset. Sus ventajas para este problema son:

- **Robusto al desbalance de clases** cuando se usa `class_weight='balanced'`.
- **No requiere normalización** de variables, lo que simplifica el preprocesamiento.
- **Maneja variables categóricas** con alta cardinalidad como `ENTIDAD` o `MUNICIPIO`
  de forma eficiente tras una codificación adecuada.
- **Proporciona importancia de variables**, lo que nos permitirá analizar qué
  atributos son más relevantes para predecir el estatus de la víctima, como
  pide la tarea.

### Consideraciones sobre el desbalance de clases

El dataset presenta un **fuerte desbalance**. Hay dos cosas importantes a considerar:

1. **Métricas**: el accuracy no es una métrica adecuada en este contexto. Un modelo
   que siempre prediga `DESAPARECIDA` obtendría un 94% de accuracy sin aprender
   nada útil. Se utilizarán **precision, recall y F1-score**, priorizando el recall
   de la clase minoritaria (`NO LOCALIZADA`), ya que en el contexto real es más
   costoso no identificar a una persona localizada que cometer un falso positivo.

2. **Manejo del desbalance**: se usará `class_weight='balanced'` en el modelo,
   lo que ajusta automáticamente los pesos de cada clase inversamente proporcional
   a su frecuencia, penalizando más los errores en la clase minoritaria.

### Selección de features

Se descartan las siguientes columnas:
- `ID_VICTIMA`: identificador único, no aporta información predictiva.
- `FECHA_NACIMIENTO`, `FECHA_DESAPARICION`, `FECHA_REGISTRO`: fechas crudas,
  ya representadas por `GRUPO_EDAD`, `SEXENIO_DESAPARICION` y `SEXENIO_REGISTRO`.
- `CVE_ENT`, `CVE_MUN`: claves numéricas de entidad y municipio, ya representadas
  por `ENTIDAD` y `MUNICIPIO`.
- `ORIGEN_REPORTE`: altamente correlacionado con `ENTIDAD`.

Las features finales son: `SEXO`, `ENTIDAD`, `MUNICIPIO`, `SEXENIO_DESAPARICION`,
`SEXENIO_REGISTRO`, `MES_DESAPARICION` y `GRUPO_EDAD`.


In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Features y variable objetivo
FEATURES = [
    "SEXO",
    "ENTIDAD",
    "MUNICIPIO",
    "SEXENIO_DESAPARICION",
    "SEXENIO_REGISTRO",
    "MES_DESAPARICION",
    "GRUPO_EDAD"
]

X = data[FEATURES].copy()
y = data["ESTATUS_VICTIMA"].copy()

# Codificación de variables categóricas
# Random Forest no trabaja directamente con strings,
# por lo que codificamos cada columna con LabelEncoder
encoders = {}
for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

print(f"Dimensiones de X: {X.shape}")
print(f"\nDistribución de y:\n{y.value_counts()}")

Dimensiones de X: (84738, 7)

Distribución de y:
ESTATUS_VICTIMA
DESAPARECIDA     79881
NO LOCALIZADA     4857
Name: count, dtype: int64


In [32]:
# División train/test
# Se usa estratificación para mantener la proporción de clases
# en ambos conjuntos, lo cual es importante dado el desbalance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]}")
print(f"Tamaño del conjunto de prueba: {X_test.shape[0]}")
print(f"\nDistribución en entrenamiento:\n{y_train.value_counts()}")
print(f"\nDistribución en prueba:\n{y_test.value_counts()}")

Tamaño del conjunto de entrenamiento: 67790
Tamaño del conjunto de prueba: 16948

Distribución en entrenamiento:
ESTATUS_VICTIMA
DESAPARECIDA     63904
NO LOCALIZADA     3886
Name: count, dtype: int64

Distribución en prueba:
ESTATUS_VICTIMA
DESAPARECIDA     15977
NO LOCALIZADA      971
Name: count, dtype: int64


In [33]:
import time

inicio = time.time()

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1  # usa todos los núcleos disponibles
)

rf.fit(X_train, y_train)

print(f"Tiempo de entrenamiento: {time.time() - inicio:.2f} segundos")

Tiempo de entrenamiento: 1.78 segundos


In [34]:
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))

               precision    recall  f1-score   support

 DESAPARECIDA       0.97      0.95      0.96     15977
NO LOCALIZADA       0.39      0.48      0.43       971

     accuracy                           0.93     16948
    macro avg       0.68      0.72      0.70     16948
 weighted avg       0.94      0.93      0.93     16948



In [35]:
import pandas as pd

importancias = pd.Series(
    rf.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print("Importancia de variables:")
print(importancias)

Importancia de variables:
ENTIDAD                 0.368853
MUNICIPIO               0.253138
MES_DESAPARICION        0.126833
SEXENIO_REGISTRO        0.123040
GRUPO_EDAD              0.069611
SEXENIO_DESAPARICION    0.044026
SEXO                    0.014500
dtype: float64


## Resultados del modelo de clasificación

### Métricas de desempeño

| Clase | Precision | Recall | F1-score | Support |
|---|---|---|---|---|
| DESAPARECIDA | 0.97 | 0.95 | 0.96 | 15,977 |
| NO LOCALIZADA | 0.39 | 0.48 | 0.43 | 971 |
| **Accuracy** | | | **0.93** | 16,948 |
| Macro avg | 0.68 | 0.72 | 0.70 | 16,948 |
| Weighted avg | 0.94 | 0.93 | 0.93 | 16,948 |

### Interpretación de las métricas

El modelo muestra un comportamiento típico ante datasets desbalanceados:

**Para la clase `DESAPARECIDA`** el modelo funciona muy bien, con un F1-score de
0.96. Esto era esperado dado que representa el 94% de los datos y el modelo tiene
abundante información para aprender sus patrones.

**Para la clase `NO LOCALIZADA`** el desempeño es considerablemente menor, con un
F1-score de 0.43. El recall de 0.48 indica que el modelo solo identifica
correctamente 1 de cada 2 casos de personas no localizadas. Esto es preocupante
en el contexto real, ya que significa que aproximadamente la mitad de las personas
no localizadas serían clasificadas erróneamente como desaparecidas.

A pesar de usar `class_weight='balanced'`, el fuerte desbalance (94%/6%) sigue
limitando la capacidad del modelo para aprender los patrones de la clase minoritaria.

### Importancia de variables

| Variable | Importancia |
|---|---|
| ENTIDAD | 0.369 |
| MUNICIPIO | 0.253 |
| MES_DESAPARICION | 0.127 |
| SEXENIO_REGISTRO | 0.123 |
| GRUPO_EDAD | 0.070 |
| SEXENIO_DESAPARICION | 0.044 |
| SEXO | 0.015 |

Las variables más relevantes para predecir el estatus de la víctima son **ENTIDAD**
y **MUNICIPIO**, que juntas explican el 62% de la importancia del modelo. Esto
sugiere que **la ubicación geográfica es el factor más determinante** para predecir
si una persona será localizada o permanecerá desaparecida.

**MES_DESAPARICION** y **SEXENIO_REGISTRO** tienen una importancia similar (~12%
cada uno), lo que indica que el período temporal también juega un papel relevante.

**SEXO** es la variable menos importante (1.4%), lo que es consistente con los
hallazgos del ejercicio 2, donde tampoco encontramos asociaciones fuertes entre
el sexo y patrones geográficos o temporales.

### ¿Captura el modelo patrones reales o sesgos?

Es importante que refelxionemos sobre si estas importancias reflejan el fenómeno real
o sesgos en los datos:

- La alta importancia de **ENTIDAD** y **MUNICIPIO** podría reflejar diferencias
  reales en la capacidad institucional de localización entre estados, pero también
  podría estar capturando sesgos en el sistema de registro, donde algunas entidades
  tienen procesos más rigurosos de actualización de estatus que otras.

- La importancia de **SEXENIO_REGISTRO** sugiere que el período administrativo
  influye en la probabilidad de localización, lo que podría reflejar cambios en
  las políticas de búsqueda entre gobiernos aunque no nos explica exactamente cómo o por qué.

- El modelo no debe interpretarse como una herramienta predictiva confiable para
  casos individuales, dado el bajo desempeño en la clase `NO LOCALIZADA`. Su valor
  principal en todo caso es identificar qué variables están asociadas al estatus de la víctima
  a nivel agregado.